<a href="https://colab.research.google.com/github/comitanigiacomo/Comitani_85673A_Stream_Analysis/blob/main/Comitani_85673A_Stream_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
# Setup Environment

!apt-get update -qq && apt-get install -y openjdk-17-jdk-headless -qq > /dev/null

%pip install -q kaggle mmh3

import os
import csv
import mmh3

current_dir = os.path.abspath(".")

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["SPARK_HOME"] = os.path.join(current_dir, "spark")
os.environ['KAGGLE_USERNAME'] = os.getenv('KAGGLE_USERNAME', "xxxxx")
os.environ['KAGGLE_KEY'] = os.getenv('KAGGLE_KEY', "xxxxx")

!test -d nytdataset && echo "Skipping" || kaggle datasets download -d "benjaminawd/new-york-times-articles-comments-2020"
!test -d nytdataset && echo "Skipping" || unzip -q -d nytdataset new-york-times-articles-comments-2020.zip
!rm new-york-times-articles-comments-2020.zip 2> /dev/null

CORRECTION = 0.77351
repo = "nytdataset/"

E: List directory /var/lib/apt/lists/partial is missing. - Acquire (13: Permission denied)
Note: you may need to restart the kernel to use updated packages.
Skipping
Skipping


In [18]:
# Dataset Streaming Interface

# Create a Python generator using yield to simulate a data stream on the dataset.
# I need to implement it like I'm working with a massive dataset, so using a function like pandas.read_csv
# would mean cheating, because all the data would be loaded into RAM.
# My goal is to create a lazy iterator using yield instead of return.

def stream_all_comments(repo):
    for i in range(10):
        filepath = f"{repo}/nyt-comments-part{i}.csv"
        
        with open(filepath, mode='r', encoding='utf-8') as file:
            reader = csv.reader(file)
            headers = next(reader)
            user_id_index = headers.index("userID")
            for row in reader:
                yield row[user_id_index]

In [ ]:
# Implementation 1

# I want to know in real time how many unique users are interacting today on the site, 
# to understand if there is a bot attack or just to monitor the level of engagement, 
# so I can figure out the best time to publish an article for maximum visibility.
# Obviously, I don't want to keep all the user IDs in RAM.

# Flajolet-Martin Algorithm

# NOTE: For better precision, I will calculate the median of the averages.

class FlajoletMartin:
    def __init__(self, num_hashes=256, num_groups=16):
        self.num_hashes = num_hashes
        self.num_groups = num_groups
        self.output_powers = [1] * self.num_hashes

    def analyze_user_ID(self, item): 
        for i in range(self.num_hashes):
            h = mmh3.hash128(str(item), i)
            
            lowest_bit_value = h & -h
            
            if lowest_bit_value > self.output_powers[i]:
                self.output_powers[i] = lowest_bit_value
        
    def get_results(self):
        results = self.output_powers
        
        n = len(results) // self.num_groups
        groups = [results[i:i + n] for i in range(0, len(results), n)]
                                
        means = []
        for group in groups:
            means.append(sum(group) / len(group))
        
        means.sort()
        m = len(means)
        mid = m // 2
        
        if m % 2 != 0:
            raw_estimate = means[mid]
        else:
            raw_estimate = (means[mid-1] + means[mid]) / 2.0
            
        return int(raw_estimate)
    

In [20]:
#repo = "nytdataset/"
#FM = FlajoletMartin()

#stream = stream_all_comments(repo)


#for id_utente in stream:
    #FM.analyze_user_ID(id_utente)
    
#print("FlajoletMartin Unique Users:", FM.get_results())

In [21]:
# Implementation 2: Idea from LogLog version

class FlajoletMartinLogLog:
    def __init__(self, num_hashes=256, num_groups=16):
        self.num_hashes = num_hashes
        self.num_groups = num_groups
        self.max_zeros = [0] * self.num_hashes

    def analyze_user_ID(self, item): 
        
        for i in range(self.num_hashes):
            h = mmh3.hash128(str(item), i)
            
            # Modification 1: Track the number of trailing zeros instead of the power of two to mitigate variance.
            zeros = (h & -h).bit_length() - 1
            
            if zeros > self.max_zeros[i]:
                self.max_zeros[i] = zeros
        
    def get_results(self):
        results = self.max_zeros
        
        n = len(results) // self.num_groups
        groups = [results[i:i + n] for i in range(0, len(results), n)]
                                
        means = []
        for group in groups:
            means.append(sum(group) / len(group))
        
        means.sort()
        m = len(means)
        mid = m // 2
        
        if m % 2 != 0:
            median_of_means = means[mid]
        else:
            median_of_means = (means[mid-1] + means[mid]) / 2.0
            
        # Modification 2: Apply the power of two to the final median, then scaled by the correction factor to adjust the result.  
        final_estimate = (2 ** median_of_means) * CORRECTION
            
        return int(final_estimate)


In [22]:
FMLL = FlajoletMartinLogLog()

stream = stream_all_comments(repo)

for id_utente in stream:
    FMLL.analyze_user_ID(id_utente)
    
print("FlajoletMartinLogLog Unique Users:", FMLL.get_results())

FlajoletMartinLogLog Unique Users: 423496


In [ ]:
# Bloom Filter

# I can use the bloom filter to categorize the comments in base of the type of document they're commenting about.

# Creation of the trusted list S:

def create_trusted_list(repo):
    with open(repo + "nyt-articles-2020.csv", mode='r', encoding='utf-8') as file:
        reader = csv.reader(file)
        headers = next(reader)
        
        article_id = headers.index("uniqueID")
        article_section = headers.index("section")
        
        for row in reader:
            if row[article_section] == "Science":
                yield row[article_id]

In [24]:
trusted_list = create_trusted_list(repo)

In [25]:
def count_trusted_articles_len(repo):
    count = 0
    with open(repo + "nyt-articles-2020.csv", mode='r', encoding='utf-8') as file:
        reader = csv.reader(file)
        headers = next(reader)
        article_section = headers.index("section")
        
        for row in reader:
            if row[article_section] == "Science":
                count += 1
    return count

In [26]:
count_trusted_articles_len("nytdataset/")

354

In [27]:
class BloomFilter:
    def __init__(self, m, k):
        self.m = m
        self.k = k
        self.bloom = [0] * m
        
    def add(self, key):
        for i in range(self.k):
            index = mmh3.hash(str(key), i) % self.m
            self.bloom[index] = 1
    
    def check(self, key):
        for i in range(self.k):
            index = mmh3.hash(str(key), i) % self.m
            
            if self.bloom[index] == 0:
                return False 
        return True

In [ ]:
m = 3540
k = 7
bf = BloomFilter(m, k)

s = create_trusted_list(repo)

for article_id in create_trusted_list(repo):
    bf.add(article_id)

count_science_comments = 0
total_processed = 0

for comment_article_id in stream_all_comments(repo):
    total_processed += 1
    
    if bf.check(comment_article_id):
        count_science_comments += 1

print(f"Total comments analyzed: {total_processed}")
print(f"Comments (probably) regarding science: {count_science_comments}")

Total comments analyzed: 4986461
Comments (probably) regarding science: 39888


In [30]:
trusted_set = set(create_trusted_list(repo)) 
count_real = 0
total = 0

for i in range(10):
    filename = f"{repo}nyt-comments-part{i}.csv"
    
    with open(filename, mode='r', encoding='utf-8') as file:
        reader = csv.reader(file)
        next(reader)
        
        for row in reader:
            if row and len(row) > 0:
                total += 1
                if row[-1] in trusted_set:
                    count_real += 1
    
print(f"Real Science comments: {count_real:,}")

Real Science comments: 23,698
